In [ ]:
# Notebook 01 — Cell 1: inspect chat-template token boundaries (no model weights needed)
from transformers import AutoTokenizer
import json
from google.colab import userdata

# Retrieve HF_TOKEN securely from Google Colab Secrets and strip any internal/external newlines or carriage returns
try:
    raw_token = userdata.get('HF_TOKEN')
    hf_token = raw_token.replace('\r', '').replace('\n', '').strip() if raw_token else None
except Exception as e:
    print("Warning: Could not retrieve HF_TOKEN from secrets. If needed, please add it to the Secrets manager.")
    hf_token = None

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token, use_fast=True)
messages = [{"role": "user", "content": "Explain why the sky appears blue."}]
rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Tokenize the rendered string to guarantee we get a list of integer token IDs
ids = tok.encode(rendered, add_special_tokens=False)

print("Rendered template:", repr(rendered))
print("Token count:", len(ids))
print("First 12 IDs:", ids[:12])
print("Last 20 tokens:", tok.convert_ids_to_tokens(ids[-20:]))
print("Decoded tail:", repr(tok.decode(ids[-20:], skip_special_tokens=False)))
print("BOS token:", tok.bos_token, "| BOS ID:", tok.bos_token_id)
print("First input ID:", ids[0])
print("Starts with BOS ID:", tok.bos_token_id is not None and ids[0] == tok.bos_token_id)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Rendered template: '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nExplain why the sky appears blue.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
Token count: 43
First 12 IDs: [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220]
Last 20 tokens: ['4', 'ĊĊ', '<|eot_id|>', '<|start_header_id|>', 'user', '<|end_header_id|>', 'ĊĊ', 'Ex', 'plain', 'Ġwhy', 'Ġthe', 'Ġsky', 'Ġappears', 'Ġblue', '.', '<|eot_id|>', '<|start_header_id|>', 'assistant', '<|end_header_id|>', 'ĊĊ']
Decoded tail: '4\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nExplain why the sky appears blue.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
BOS token: <|begin_of_text|> | BOS ID: 128000
First input ID: 128000
Starts with BOS ID: True


In [ ]:
# Notebook 01 — Cell 2: identify user-content token span using a sentinel test
# This is a diagnostic, not a universal offset algorithm: chat-template tokenization can merge
# boundaries. Compare template with and without the sentinel and inspect the tokenized text.
sentinel = "UNIQUE_SENTINEL_7319"
msgs = [{"role": "user", "content": f"before {sentinel} after"}]

# Generate the rendered template string first
rendered_sentinel = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

# Encode the template string to guarantee a clean list of integer token IDs
ids = tok.encode(rendered_sentinel, add_special_tokens=False)

decoded_tokens = [tok.convert_ids_to_tokens([i])[0] for i in ids]
print(list(enumerate(decoded_tokens)))
print("Full decoded:", repr(tok.decode(ids, skip_special_tokens=False)))
print("Sentinel tokenized standalone:", tok.tokenize(sentinel))

[(0, '<|begin_of_text|>'), (1, '<|start_header_id|>'), (2, 'system'), (3, '<|end_header_id|>'), (4, 'ĊĊ'), (5, 'Cut'), (6, 'ting'), (7, 'ĠKnowledge'), (8, 'ĠDate'), (9, ':'), (10, 'ĠDecember'), (11, 'Ġ'), (12, '202'), (13, '3'), (14, 'Ċ'), (15, 'Today'), (16, 'ĠDate'), (17, ':'), (18, 'Ġ'), (19, '26'), (20, 'ĠJul'), (21, 'Ġ'), (22, '202'), (23, '4'), (24, 'ĊĊ'), (25, '<|eot_id|>'), (26, '<|start_header_id|>'), (27, 'user'), (28, '<|end_header_id|>'), (29, 'ĊĊ'), (30, 'before'), (31, 'ĠUNIQUE'), (32, '_SENT'), (33, 'IN'), (34, 'EL'), (35, '_'), (36, '731'), (37, '9'), (38, 'Ġafter'), (39, '<|eot_id|>'), (40, '<|start_header_id|>'), (41, 'assistant'), (42, '<|end_header_id|>'), (43, 'ĊĊ')]
Full decoded: '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nbefore UNIQUE_SENTINEL_7319 after<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
Sentinel to

In [ ]:
import pandas as pd
import numpy as np

# Initialize a list to store our template token records
template_records = []

def record_template_analysis(
    model_id,
    input_ids,
    last_prefill_index=None,
    method="sentinel_alignment",
    offset_status="resolved"
):
    """
    Records a single template analysis entry with structural details.
    """
    record = {
        "model_id": model_id,
        "input_ids": list(input_ids),
        "token_count": len(input_ids),
        "last_prefill_index": last_prefill_index,
        "method": method,
        "offset_status": offset_status
    }

    template_records.append(record)
    return pd.DataFrame(template_records)

# Demonstration with Cell 2 data
# Let's find the last prefill index (the index before the <|start_header_id|> assistant tag, which is at index 40)
try:
    # Find the index of '<|start_header_id|>assistant' structure or the end of the user message
    assistant_header_idx = ids.index(tok.convert_tokens_to_ids('<|start_header_id|>'))
    # In Llama 3.1, assistant starts after the system/user block, let's locate the last header sequence
    # Index of '<|start_header_id|>' before assistant is at 40
    last_prefill = 39 # Index of the token before <|start_header_id|> assistant
    status = "resolved"
except Exception:
    last_prefill = None
    status = "unresolved"

# Store the record
df_records = record_template_analysis(
    model_id=MODEL_ID,
    input_ids=ids,
    last_prefill_index=last_prefill,
    method="sentinel_alignment",
    offset_status=status
)

display(df_records)

,model_id,input_ids,token_count,last_prefill_index,method,offset_status
0,meta-llama/Llama-3.1-8B-Instruct,"[128000, 128006, 9125, 128007, 271, 38766, 130...",44,39,sentinel_alignment,resolved
